# Lesson 14: Parameter-Efficient Fine-Tuning (LoRA & QLoRA) and Quantized Edge Export

In this companion notebook, you will:
1. Configure a **PEFT LoRA** adapter on a compact open-source SLM (`HuggingFaceTB/SmolLM2-135M-Instruct` or `Qwen/Qwen2.5-0.5B-Instruct`).
2. Inspect trainable vs. frozen parameter counts (typically < 1.5% trainable).
3. Merge LoRA weights back into the base linear layers (`merge_and_unload()`) so edge inference incurs **zero extra adapter latency**.

In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes torch

## 1. Attach LoRA Adapters & Inspect Trainable Parameters
We inject rank $r=8, \alpha=16$ low-rank matrices into `q_proj` and `v_proj`.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id)

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(base_model, lora_cfg)
peft_model.print_trainable_parameters()

## 2. Merge LoRA Weights for Zero-Overhead Edge Export
After fine-tuning, `merge_and_unload()` folds $\frac{\alpha}{r} B A$ directly into $W_0$ so the exported model is a clean standard Hugging Face / ONNX / GGUF checkpoint.

In [ ]:
merged_model = peft_model.merge_and_unload()
save_dir = "./merged_edge_slm"
merged_model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Merged edge-ready model saved to {save_dir}")